In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

# Progress bar
from tqdm.auto import tqdm
tqdm.pandas()

pd.set_option('display.max_columns', None)

In [3]:
from news_featuring import extract_news_features_pipeline
from news_featuring import floor_or_ceil
from news_featuring import aggregate_events

# Load and preprocess events

> Upload

In [4]:
PATH_TO_NEWS = 'economic_events/economic_events.csv'
news = pd.read_csv(PATH_TO_NEWS)
news = news[['date', 'currency', 'importance', 'title', 'indicator', 'country', 'category', 'actual', 'forecast', 'previous', 
             'comment', 'source', 'source_url', 'referenceDate', 'period', 'unit', 'scale']]
news['date'] = pd.to_datetime(news['date'], utc=True)
news.sort_values(by='date', ascending=True, inplace=True)

In [5]:
print(f'Start date: {news['date'].dt.date.min()}')
print(f'End date: {news['date'].dt.date.max()}')
print(f'Count of rows: {news.shape[0]}')
print(f'Shape: {news.shape}')

Start date: 2013-01-04
End date: 2026-03-30
Count of rows: 99029
Shape: (99029, 17)


In [6]:
def preprocess_events(news: pd.DataFrame, datetime_crop_method: str = '1st'):
    news = extract_news_features_pipeline(news)

    print('[INFO] Cropping datetime hours...')
    if datetime_crop_method == '1st':
        news['date'] = pd.to_datetime(news['date'], utc=True)
        news['rounded_time'] = news['date'].apply(lambda x: x.floor('1h'))

    elif datetime_crop_method == '2nd':
        news['date'] = pd.to_datetime(news['date'], utc=True)
        news['rounded_time'] = news['date'].apply(lambda x: floor_or_ceil(x, freq='h'))
    
    print('[INFO] Aggregating events...')
    agg_events = aggregate_events(news, dt_col='rounded_time')
    agg_events['time_to_check'] = agg_events['rounded_time'] - pd.Timedelta(hours=1)
    print('[INFO] Done!')

    return agg_events

In [7]:
agg_events = preprocess_events(news, '1st')

[INFO] Category dummies added.
[INFO] Currency dummies added.
[INFO] Country dummies added.
[INFO] Source dummies added.
[INFO] Event category dummies added.
[INFO] Stage release dummies added.
[INFO] Event calculation period dummies added.
[INFO] Scale dummies added.
[INFO] Most important event dummies added.
[INFO] Flag "is_calendar" added.
[INFO] Flag "is_president" added.
[INFO] Flag "is_election" added.
[INFO] Removed "OTHER" and redundant columns.
[INFO] Cropping datetime hours...
[INFO] Aggregating events...
[INFO] Done!


## Load prices

In [8]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

path = os.path.join('..', 'dataset', 'mt5_tw_h1')
files = os.listdir(path)
pairs = [p.split('_')[0] for p in files]

In [9]:
from price_featuring import add_features, get_base_and_quote_currency
from targets import set_targets

In [10]:
def add_features_and_targets_to_prices(prices: pd.DataFrame, period: int, N: int = 8):
    prices = add_features(prices, period=period)
    prices = set_targets(prices, look_forward_bars=N)
    return prices

In [11]:
PERIOD = 21
LOOK_FORWARD = 4

> Load and preprocessing prices

In [12]:
print(pairs)

['AUDCAD', 'AUDUSD', 'EURAUD', 'EURCAD', 'EURGBP', 'EURUSD', 'GBPCHF', 'GBPUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY']


In [13]:
prices = []
for pair in (pbar := tqdm(pairs, desc="Reading files", total=len(pairs))):
    pbar.set_description(f"Reading file: {pair}")
    pair_prices = pd.read_csv(os.path.join(path, f'{pair}_H1.csv'))
    pair_prices.rename(columns={'datetime': 'time'}, inplace=True)
    pair_prices['instrument'] = pair
    pair_prices[['base_currency', 'quote_currency']] = get_base_and_quote_currency(pair)
    pair_prices = add_features_and_targets_to_prices(pair_prices, period=PERIOD, N=8)
    pair_prices.drop(['open', 'high', 'low', 'close', 'realized_vol_long', 'realized_vol_short'], axis=1, inplace=True)
    prices.append(pair_prices)

prices = pd.concat(prices, ignore_index=True)

Reading file: USDJPY: 100%|██████████| 12/12 [00:04<00:00,  2.69it/s]


In [14]:
prices.shape

(1188197, 49)

In [15]:
prices.head()

,time,instrument,base_currency,quote_currency,daily_atr,trange,atr,kaufman_efficiency_ratio,custom_efficiency_ratio,wick_ratio,relative_range,relative_atr,normalized_bb_width,distance_from_sma,ADX_21,ADXR_21_2,DMP_21,DMN_21,di_spread,tr_over_atr,realized_vol_ratio,parkinson_vol,parkinson_vol_over_atr,atr_short_over_long,abs_log_return_sum,vvov,week,month,quarter,dayofweek,day,hour,log_return,trg_future_range_1h,trg_future_range_3h,trg_future_range_6h,trg_future_range_24h,trg_overall_future_range_1h,trg_overall_future_range_3h,trg_overall_future_range_6h,trg_overall_future_range_24h,trg_big_doji,trg_dir_changes,trg_is_flat_flg,trg_is_trend_flg,trg_is_chaos_1h,trg_is_chaos_3h,trg_is_chaos_6h,trg_is_chaos_24h
0,2010-05-07 22:00:00+00:00,AUDCAD,AUD,CAD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18,5,2,4,7,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0,0,False,False,False,False
1,2010-05-10 00:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00319,NaN,NaN,NaN,0.087774,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,0,0.001251,NaN,NaN,NaN,NaN,1.543815,NaN,NaN,NaN,0,2.0,0,0,False,False,False,False
2,2010-05-10 01:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00183,NaN,NaN,NaN,0.311475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,1,0.000636,NaN,NaN,NaN,NaN,1.741669,NaN,NaN,NaN,0,2.0,0,0,True,False,False,False
3,2010-05-10 02:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00305,NaN,NaN,NaN,0.475410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,2,-0.001596,NaN,NaN,NaN,NaN,1.646529,NaN,NaN,NaN,0,1.0,0,0,True,False,False,False
4,2010-05-10 03:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00411,NaN,NaN,NaN,0.615572,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,3,0.002737,NaN,NaN,NaN,NaN,1.541711,NaN,NaN,NaN,0,0.0,0,0,False,False,False,False


## Add log features

In [16]:
for trg in tqdm(['trg_future_range_1h', 'trg_future_range_3h', 'trg_future_range_6h', 'trg_future_range_24h']):
    prices[trg + '_log'] = prices[trg].apply(lambda x: np.log1p(x))

100%|██████████| 4/4 [00:04<00:00,  1.17s/it]


## Join prices to events

In [17]:
df = agg_events.merge(prices, left_on='time_to_check', right_on='time', how='left')

In [18]:
df.shape

(389482, 137)

In [19]:
def add_lagged_target_features(df, windows=[5, 20]):
    """
    Adds lagged target features for news titles and categories to capture regime shifts.
    """
    # We need to sort by date to avoid leakage
    df = df.sort_values('time_to_check')
    
    # Collect new columns in a dict to concat once per horizon
    new_cols = {}
    
    # 1. Last N events of the same dominant_event_type
    for h in tqdm([1, 3, 6, 24]):
        col = f'trg_future_range_{h}h'
        for w in windows:
            new_cols[f'prev5_avg_{h}h_{w}'] = (
                df.groupby(['main_event', 'instrument'])[col]
                .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).mean())
            ).fillna(0)
            new_cols[f'prev5_min_{h}h_{w}'] = (
                df.groupby(['main_event', 'instrument'])[col]
                .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).min())
            ).fillna(0)
            new_cols[f'prev5_max_{h}h_{w}'] = (
                df.groupby(['main_event', 'instrument'])[col]
                .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).max())
            ).fillna(0)
        
        new_cols[f'prev_{h}h'] = (
            df.groupby(['main_event', 'instrument'])[col]
            .transform(lambda x: x.shift(1))
        ).fillna(0)
    
        # 2. Overall recent volatility (last 50 news events)
        # Captures the general market regime independent of the news type
        # new_cols[f'lag_target_overall_50_{h}h'] = df[col].shift(1).rolling(50, min_periods=1).mean().fillna(0)
    
    # Concatenate all new columns at once
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df

In [20]:
df = add_lagged_target_features(df)

100%|██████████| 4/4 [00:03<00:00,  1.20it/s]


In [21]:
for ftr in ['trg_big_doji', 'trg_dir_changes', 'trg_is_flat_flg', 'trg_is_trend_flg']:
    df[ftr] = df[ftr].fillna(0).astype(int)

In [22]:
df.head()

,rounded_time,news_count,high_impact_count,key_event_count,main_event,prev_hour_news_count,next_hour_news_count,prev_hour_high_impact_count,next_hour_high_impact_count,prev_hour_main_event,next_hour_main_event,category_bnd,category_bsnss,category_cnsm,category_enrg,category_gdp,category_gov,category_hse,category_lbr,category_mny,category_mrkt,category_prce,category_trd,currency_AUD,currency_CAD,currency_EUR,currency_GBP,currency_JPY,currency_SEK,currency_SGD,currency_USD,country_AU,country_CA,country_DE,country_EU,country_FR,country_GB,country_JP,country_SE,country_SG,country_US,source_CENTRAL_BANK,source_ENERGY,source_GOV_MINISTRY,source_INDUSTRY,source_OFFICES,source_OFFICIAL_STATS,source_PRIVATE_SURVEY,source_RATING_AGENCY,source_RESEARCH_INSTITUTES,source_US_HOUSING,event_MONETARY_POLICY,event_CB_SPEECH,event_INFLATION,event_LABOR_MARKET,event_ECONOMIC_ACTIVITY,event_SENTIMENT,event_CONSUMER_HOUSING,event_TRADE_FINANCE,event_COMMODITIES,stage_release_Final,stage_release_Flash,stage_release_Preliminary,calc_period_MoM,calc_period_QoQ,calc_period_YoY,is_calendar,is_president,mie_Balance_of_Trade,mie_Core_Inflation_rate,mie_FOMC,mie_GDP,mie_Inflation_rate,mie_Interest_Rate_Decision,mie_NFP,mie_PMI,mie_PMI_Manufacturing,mie_PMI_Services,mie_Retail_Sales,mie_Unemployment_rate,time_passed_from_last_events,time_left_to_next_events,last_important_event_in_hours,time_to_check,time,instrument,base_currency,quote_currency,daily_atr,trange,atr,kaufman_efficiency_ratio,custom_efficiency_ratio,wick_ratio,relative_range,relative_atr,normalized_bb_width,distance_from_sma,ADX_21,ADXR_21_2,DMP_21,DMN_21,di_spread,tr_over_atr,realized_vol_ratio,parkinson_vol,parkinson_vol_over_atr,atr_short_over_long,abs_log_return_sum,vvov,week,month,quarter,dayofweek,day,hour,log_return,trg_future_range_1h,trg_future_range_3h,trg_future_range_6h,trg_future_range_24h,trg_overall_future_range_1h,trg_overall_future_range_3h,trg_overall_future_range_6h,trg_overall_future_range_24h,trg_big_doji,trg_dir_changes,trg_is_flat_flg,trg_is_trend_flg,trg_is_chaos_1h,trg_is_chaos_3h,trg_is_chaos_6h,trg_is_chaos_24h,trg_future_range_1h_log,trg_future_range_3h_log,trg_future_range_6h_log,trg_future_range_24h_log,prev5_avg_1h_5,prev5_min_1h_5,prev5_max_1h_5,prev5_avg_1h_20,prev5_min_1h_20,prev5_max_1h_20,prev_1h,prev5_avg_3h_5,prev5_min_3h_5,prev5_max_3h_5,prev5_avg_3h_20,prev5_min_3h_20,prev5_max_3h_20,prev_3h,prev5_avg_6h_5,prev5_min_6h_5,prev5_max_6h_5,prev5_avg_6h_20,prev5_min_6h_20,prev5_max_6h_20,prev_6h,prev5_avg_24h_5,prev5_min_24h_5,prev5_max_24h_5,prev5_avg_24h_20,prev5_min_24h_20,prev5_max_24h_20,prev_24h
0,2013-01-04 13:00:00+00:00,2,1,2,NFP,0.0,1.0,0.0,0.0,None,Inflation_rate,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,2,0,0,0,0,1,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0.0,96.0,NaN,2013-01-04 12:00:00+00:00,2013-01-04 12:00:00+00:00,AUDCAD,AUD,CAD,0.005046,0.00158,0.001374,0.166421,4.201550,0.132911,0.132875,-0.333868,6.797432,2.028329,14.055974,13.829393,0.004277,0.006419,-0.002143,1.149523,1.031834,0.000795,0.578545,0.965506,0.004959,0.027104,1,1.0,1.0,4.0,4.0,12.0,0.000233,3.077522,3.077522,5.813098,6.053188,1.231387,2.609312,2.464022,6.878981,0,2,0,1,False,False,False,False,1.405490,1.405490,1.918847,1.953480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,2013-01-04 13:00:00+00:00,2,1,2,NFP,0.0,1.0,0.0,0.0,None,Inflation_rate,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,2,0,0,0,0,1,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0.0,96.0,NaN,2013-01-04 12:00:00+00:00,2013-01-04 12:00:00+00:00,USDJPY,USD,JPY,0.614801,0.09300,0.157854,0.361219,2.382838,0.139785,-0.669817,0.949123,17.107682,20.580434,39.638943,38.252693,1.299465,0.254437,1.045027,0.589152,1.202590,0.001342,0.008502,1.057840,0.006875,0.053198,1,1.0,1.0,4.0,4.0,12.0,0.000181,3.655276,5.061639,5.061639,7.354897,1.296358,2.244052,3.325403,7.645988,0,3,0,1

In [23]:
df.shape

(389482, 165)

In [24]:
df.isna().sum().sort_values(ascending=False)

wick_ratio                  1210
trg_future_range_24h_log    1201
trg_future_range_6h_log     1201
trg_future_range_3h_log     1201
trg_is_chaos_1h             1201
                            ... 
prev5_max_24h_5                0
prev5_avg_24h_20               0
prev5_min_24h_20               0
prev5_max_24h_20               0
prev_24h                       0
Length: 165, dtype: int64

In [25]:
trg_columns = [c for c in df.columns if c.startswith('trg_')]
df.dropna(subset=trg_columns+['wick_ratio'], inplace=True)

In [26]:
df.isna().sum().sort_values(ascending=False)

next_hour_main_event             12
prev_hour_main_event             12
last_important_event_in_hours    12
news_count                        0
high_impact_count                 0
                                 ..
prev5_max_24h_5                   0
prev5_avg_24h_20                  0
prev5_min_24h_20                  0
prev5_max_24h_20                  0
prev_24h                          0
Length: 165, dtype: int64

In [27]:
df.dropna(subset=['prev_hour_main_event', 'next_hour_main_event', 'last_important_event_in_hours'], inplace=True)

# Train models

In [28]:
SUFFX = 'WITH LAGS (no log) AND NO DAILY ATR'

In [29]:
df_full = df.copy()

In [30]:
df.drop('daily_atr', axis=1, inplace=True)

In [31]:
FEATURES = [i for i in df.columns if not i.startswith('trg') and i not in ['time', 'time_to_check', 'rounded_time']]
CATEGORICAL_FEATURES = ['main_event', 'instrument', 'base_currency', 'quote_currency', 'prev_hour_main_event', 'next_hour_main_event']

In [32]:
df[CATEGORICAL_FEATURES] = df[CATEGORICAL_FEATURES].fillna("missing").astype(str)

for cat_feat in CATEGORICAL_FEATURES:
    df[cat_feat] = df[cat_feat].astype('category')

In [33]:
print(f'TARGETS: {[i for i in df.columns if i.startswith('trg')]}')

TARGETS: ['trg_future_range_1h', 'trg_future_range_3h', 'trg_future_range_6h', 'trg_future_range_24h', 'trg_overall_future_range_1h', 'trg_overall_future_range_3h', 'trg_overall_future_range_6h', 'trg_overall_future_range_24h', 'trg_big_doji', 'trg_dir_changes', 'trg_is_flat_flg', 'trg_is_trend_flg', 'trg_is_chaos_1h', 'trg_is_chaos_3h', 'trg_is_chaos_6h', 'trg_is_chaos_24h', 'trg_future_range_1h_log', 'trg_future_range_3h_log', 'trg_future_range_6h_log', 'trg_future_range_24h_log']


## Quantile ranges
- predict ranges for the next 1, 3, 6, 24 hours

In [34]:
from training_functions import run_time_series_cv_catboost_quantile_gpu

In [35]:
DESC = f"""
{SUFFX}
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


WITH LAGS (no log) AND NO DAILY ATR
- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [36]:
EXPERIMENT_NAME = 'Target: Total Range Prediction || TF: 1h || Price source: MT5 + TW || [Time cropping: 1st method]'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_future_range_{hour}h'
    RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_catboost_quantile_gpu(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour


2026/04/28 23:29:21 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/04/28 23:29:21 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/04/28 23:29:21 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/04/28 23:29:21 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/04/28 23:29:21 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/04/28 23:29:21 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/04/28 23:29:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/28 23:29:21 INFO mlflow.store.db.utils: Updating database tables
2026/04/28 23:29:21 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/04/28 23:29:21 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/04/28 23:29:21 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/04/28 23:29:21 INFO alembic.runtime


Training final GPU quantile models on all data...
[INFO] Train model for 3 hour



Training final GPU quantile models on all data...
[INFO] Train model for 6 hour



Training final GPU quantile models on all data...
[INFO] Train model for 24 hour



Training final GPU quantile models on all data...


In [37]:
EXPERIMENT_NAME = 'Target: Log Total Range Prediction || TF: 1h || Price source: MT5 + TW || [Time cropping: 1st method]'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_future_range_{hour}h_log'
    RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_catboost_quantile_gpu(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'log_range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour



Training final GPU quantile models on all data...
[INFO] Train model for 3 hour



Training final GPU quantile models on all data...
[INFO] Train model for 6 hour



Training final GPU quantile models on all data...
[INFO] Train model for 24 hour



Training final GPU quantile models on all data...


In [38]:
EXPERIMENT_NAME = 'Target: Sum of Ranges Prediction || TF: 1h || Price source: MT5 + TW || [Time cropping: 1st method]'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_overall_future_range_{hour}h'
    RUN_NAME = f'trg_overall_future_range_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_catboost_quantile_gpu(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'sum_range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour



Training final GPU quantile models on all data...
[INFO] Train model for 3 hour



Training final GPU quantile models on all data...
[INFO] Train model for 6 hour



Training final GPU quantile models on all data...
[INFO] Train model for 24 hour



Training final GPU quantile models on all data...


## Direction changes count

In [39]:
from training_functions import run_time_series_cv_catboost

In [40]:
DESC = f"""
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [41]:
EXPERIMENT_NAME = 'Target: Direction Count || TF: 1h || Price source: MT5 + TW || [Time cropping: 1st method]'

print(f'[INFO] Train model for Direction Count')
TARGET = f'trg_dir_changes'
RUN_NAME = f'trg_direction_count_{SUFFX}'
metrics = run_time_series_cv_catboost(
    df=df,
    task_type='ordinal',
    features=FEATURES,
    cat_features=CATEGORICAL_FEATURES,
    target=TARGET,
    time_col='time_to_check',
    experiment_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    description=DESC,
    model_params={
        'iterations': 5000,
        'learning_rate': 0.05
    },
    save_model=True,
    model_name=f'dir_changes_model_{hour}h'
)

[INFO] Train model for Direction Count



Training final model on all data...
0:	learn: 1.1240543	total: 44.3ms	remaining: 3m 41s
100:	learn: 1.0129673	total: 4.38s	remaining: 3m 32s
200:	learn: 1.0069564	total: 8.65s	remaining: 3m 26s
300:	learn: 1.0027497	total: 12.9s	remaining: 3m 22s
400:	learn: 0.9995898	total: 17.3s	remaining: 3m 18s
500:	learn: 0.9968382	total: 21.6s	remaining: 3m 13s
600:	learn: 0.9944186	total: 25.9s	remaining: 3m 9s
700:	learn: 0.9921698	total: 30.2s	remaining: 3m 4s
800:	learn: 0.9898812	total: 34.5s	remaining: 3m
900:	learn: 0.9878000	total: 38.8s	remaining: 2m 56s
1000:	learn: 0.9857543	total: 43.1s	remaining: 2m 52s
1100:	learn: 0.9837039	total: 47.4s	remaining: 2m 47s
1200:	learn: 0.9818309	total: 51.7s	remaining: 2m 43s
1300:	learn: 0.9799090	total: 56s	remaining: 2m 39s
1400:	learn: 0.9780150	total: 1m	remaining: 2m 34s
1500:	learn: 0.9762474	total: 1m 4s	remaining: 2m 30s
1600:	learn: 0.9745240	total: 1m 8s	remaining: 2m 26s
1700:	learn: 0.9728133	total: 1m 13s	remaining: 2m 22s
1800:	learn:

## Chaos prediction

In [42]:
from training_functions import run_time_series_cv_catboost

In [43]:
DESC = f"""
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [44]:
EXPERIMENT_NAME = 'Target: Chaos prediction || TF: 1h || Price source: MT5 + TW || [Time cropping: 1st method]'

print(f'[INFO] Train model for Chaos Prediction')
hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_is_chaos_{hour}h'
    RUN_NAME = f'trg_is_chaos_{hour}h_{SUFFX}'
    df[TARGET] = df[TARGET].astype(int)
    metrics = run_time_series_cv_catboost(
        df=df,
        task_type='classification',
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        save_model=True,
        model_name=f'chaos_prediction_model_{hour}h'
    )

[INFO] Train model for Chaos Prediction
[INFO] Train model for 1 hour



Training final model on all data...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 55.6ms	remaining: 55.6s
100:	total: 5.48s	remaining: 48.8s
200:	total: 10.8s	remaining: 43.1s
300:	total: 16.2s	remaining: 37.5s
400:	total: 21.5s	remaining: 32.1s
500:	total: 26.8s	remaining: 26.7s
600:	total: 32.2s	remaining: 21.3s
700:	total: 37.5s	remaining: 16s
800:	total: 42.8s	remaining: 10.6s
900:	total: 48.2s	remaining: 5.29s
999:	total: 53.5s	remaining: 0us

Final Model Metrics (on full dataset):
  Accuracy:  0.7570
  AUC:       0.7095

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.756438  0.999361  0.861095  292673.000000
1             1   0.882093  0.014638  0.028798   95575.000000
2      accuracy   0.756952  0.756952  0.756952       0.756952
3     macro avg   0.819266  0.506999  0.444946  388248.000000
4  weighted avg   0.787371  0.756952  0.656208  388248.000000
[INFO] Train model for 3 hour



Training final model on all data...


...

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 57.4ms	remaining: 57.4s
100:	total: 5.54s	remaining: 49.3s
200:	total: 10.9s	remaining: 43.3s
300:	total: 16.2s	remaining: 37.6s
400:	total: 21.7s	remaining: 32.5s
500:	total: 27s	remaining: 26.9s
600:	total: 32.2s	remaining: 21.4s
700:	total: 37.4s	remaining: 16s
800:	total: 42.7s	remaining: 10.6s
900:	total: 47.9s	remaining: 5.26s
999:	total: 53.1s	remaining: 0us

Final Model Metrics (on full dataset):
  Accuracy:  0.8062
  AUC:       0.8226

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.813817  0.965449  0.883172  294581.000000
1             1   0.737545  0.305358  0.431901   93667.000000
2      accuracy   0.806199  0.806199  0.806199       0.806199
3     macro avg   0.775681  0.635404  0.657536  388248.000000
4  weighted avg   0.795416  0.806199  0.774300  388248.000000
[INFO] Train model for 24 hour



Training final model on all data...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 52.7ms	remaining: 52.6s
100:	total: 5.26s	remaining: 46.8s
200:	total: 10.6s	remaining: 42s
300:	total: 15.8s	remaining: 36.8s
400:	total: 21.2s	remaining: 31.6s
500:	total: 26.5s	remaining: 26.4s
600:	total: 31.9s	remaining: 21.2s
700:	total: 37.2s	remaining: 15.9s
800:	total: 42.6s	remaining: 10.6s
900:	total: 47.9s	remaining: 5.26s
999:	total: 53.2s	remaining: 0us

Final Model Metrics (on full dataset):
  Accuracy:  0.8539
  AUC:       0.9061

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.865854  0.951468  0.906644  289499.000000
1             1   0.799641  0.567844  0.664097   98749.000000
2      accuracy   0.853895  0.853895  0.853895       0.853895
3     macro avg   0.832747  0.759656  0.785371  388248.000000
4  weighted avg   0.849013  0.853895  0.844954  388248.000000


## Trend / Flat prediction

In [45]:
from training_functions import run_time_series_cv_catboost

In [46]:
DESC = f"""
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [47]:
EXPERIMENT_NAME = 'Target: Flat / Trend prediction || TF: 1h || Price source: MT5 + TW [Time cropping: 1st method]'

print(f'[INFO] Train model for Flat / Trend prediction')
regime = ['trg_is_flat_flg', 'trg_is_trend_flg']
for r in regime:
    print(f'[INFO] Train model for {r}')
    TARGET = r
    RUN_NAME = f'trg_{r}_prediction_{SUFFX}'
    df[TARGET] = df[TARGET].astype(int)
    metrics = run_time_series_cv_catboost(
        df=df,
        task_type='classification',
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        }
    )

[INFO] Train model for Flat / Trend prediction
[INFO] Train model for trg_is_flat_flg



Training final model on all data...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 55.6ms	remaining: 55.6s
100:	total: 5.5s	remaining: 48.9s
200:	total: 11s	remaining: 43.8s
300:	total: 16.5s	remaining: 38.2s
400:	total: 21.8s	remaining: 32.6s
500:	total: 27.3s	remaining: 27.2s
600:	total: 32.6s	remaining: 21.7s
700:	total: 37.9s	remaining: 16.2s
800:	total: 43.1s	remaining: 10.7s
900:	total: 48.7s	remaining: 5.34s
999:	total: 53.9s	remaining: 0us

Final Model Metrics (on full dataset):
  Accuracy:  0.8389
  AUC:       0.9229

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.835686  0.879987  0.857265  213427.000000
1             1   0.843346  0.788767  0.815144  174821.000000
2      accuracy   0.838912  0.838912  0.838912       0.838912
3     macro avg   0.839516  0.834377  0.836204  388248.000000
4  weighted avg   0.839135  0.838912  0.838299  388248.000000
[INFO] Train model for trg_is_trend_flg



Training final model on all data...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 57ms	remaining: 57s
100:	total: 5.4s	remaining: 48.1s
200:	total: 10.8s	remaining: 43.1s
300:	total: 16.4s	remaining: 38.2s
400:	total: 21.7s	remaining: 32.4s
500:	total: 27s	remaining: 26.9s
600:	total: 32.3s	remaining: 21.4s
700:	total: 37.7s	remaining: 16.1s
800:	total: 42.9s	remaining: 10.7s
900:	total: 48.4s	remaining: 5.32s
999:	total: 53.6s	remaining: 0us

Final Model Metrics (on full dataset):
  Accuracy:  0.8675
  AUC:       0.9427

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.857094  0.924779  0.889651  224234.000000
1             1   0.884714  0.789195  0.834229  164014.000000
2      accuracy   0.867502  0.867502  0.867502       0.867502
3     macro avg   0.870904  0.856987  0.861940  388248.000000
4  weighted avg   0.868762  0.867502  0.866238  388248.000000
